# Session 20 — Text Representation: Turning Words into Numbers

**Goal of this notebook:** take clean text (from Session 19) and convert it into the **numeric vectors** a machine-learning model needs.

We will build, by hand and with scikit-learn, the three classic representations:

1. **One-Hot Encoding** — one slot per word
2. **Bag of Words (BoW)** — count words per document
3. **TF-IDF** — weight words by how informative they are

Then we'll see *why* TF-IDF is smarter, and where all three fall short.

> Pairs with the Session 20 slides. Everything here runs top-to-bottom with no errors.

## 0. Setup

We only need `scikit-learn`, `pandas`, and `numpy`. They come with Anaconda; if needed, uncomment the install line.

In [1]:
# !pip install scikit-learn pandas numpy

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option("display.max_columns", 50)
print("Ready. scikit-learn is set up.")

Ready. scikit-learn is set up.


---
## 1. The core problem: models do math, not words

A machine-learning model multiplies inputs by weights and adds them up. That only works on **numbers**.

In [2]:
word = "cat"

# A model would need to do something like:  weight * word  ... which is impossible:
try:
    result = 0.5 * word
except TypeError as e:
    print("Error:", e)

print("\nThe word 'cat' is just text — there is nothing to multiply.")
print("Our job: turn each document into a list (vector) of numbers.")

Error: can't multiply sequence by non-int of type 'float'

The word 'cat' is just text — there is nothing to multiply.
Our job: turn each document into a list (vector) of numbers.


We will use this tiny **corpus** (a collection of documents) throughout the notebook.
Keeping it small lets us read every number by eye.

In [3]:
corpus = [
    "the cat sat",
    "the dog ran",
    "cat and dog",
]

for i, doc in enumerate(corpus):
    print(f"Document {i}: {doc!r}")

Document 0: 'the cat sat'
Document 1: 'the dog ran'
Document 2: 'cat and dog'


---
## 2. One-Hot Encoding — one slot per word

**Idea:** fix a vocabulary, give every word its own position, and represent a word as all `0`s with a single `1`.

Let's first do it **by hand** so the mechanic is obvious.

In [4]:
# Build the vocabulary: every unique word, sorted for a stable order
vocab = sorted(set(" ".join(corpus).split()))
print("Vocabulary:", vocab)

word_to_index = {w: i for i, w in enumerate(vocab)}
print("Word -> position:", word_to_index)

def one_hot(word, vocab):
    vec = np.zeros(len(vocab), dtype=int)   # start with all zeros
    vec[word_to_index[word]] = 1            # flip the word's own slot to 1
    return vec

for w in ["cat", "dog", "the"]:
    print(f"{w:>4} -> {one_hot(w, vocab)}")

Vocabulary: ['and', 'cat', 'dog', 'ran', 'sat', 'the']
Word -> position: {'and': 0, 'cat': 1, 'dog': 2, 'ran': 3, 'sat': 4, 'the': 5}
 cat -> [0 1 0 0 0 0]
 dog -> [0 0 1 0 0 0]
 the -> [0 0 0 0 0 1]


Each word is a vector the length of the whole vocabulary. Notice that **every word is equally far from every other** — `cat` is no closer to `dog` than to `the`. One-hot carries no meaning.

scikit-learn can produce the same thing at the *document* level with `binary=True` (a slot is `1` if the word is present, regardless of how many times).

In [5]:
onehot_vec = CountVectorizer(binary=True)
onehot_matrix = onehot_vec.fit_transform(corpus)

df_onehot = pd.DataFrame(
    onehot_matrix.toarray(),
    columns=onehot_vec.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)
print("One-hot (presence/absence) per document:")
df_onehot

One-hot (presence/absence) per document:


,and,cat,dog,ran,sat,the
doc0,0,1,0,0,1,1
doc1,0,0,1,1,0,1
doc2,1,1,1,0,0,0


**Limitation in one line:** the vector length equals the vocabulary size. With 50,000 real words, every vector is 50,000 numbers — almost all zeros (*sparse*), and a brand-new word has no slot at all (*out-of-vocabulary*).

---
## 3. Bag of Words (BoW) — count the words

**Idea:** describe a whole document by **how many times** each vocabulary word appears. Order is thrown away — hence "bag".

scikit-learn's `CountVectorizer` does exactly this.

In [6]:
bow_vec = CountVectorizer()           # default: counts, not just presence
bow_matrix = bow_vec.fit_transform(corpus)

print("Vocabulary (column order):", list(bow_vec.get_feature_names_out()))
print("Matrix shape (documents x words):", bow_matrix.shape)

Vocabulary (column order): ['and', 'cat', 'dog', 'ran', 'sat', 'the']
Matrix shape (documents x words): (3, 6)


The result is a **document-term matrix**: one row per document, one column per word.

In [7]:
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vec.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)
print("Bag of Words counts:")
df_bow

Bag of Words counts:


,and,cat,dog,ran,sat,the
doc0,0,1,0,0,1,1
doc1,0,0,1,1,0,1
doc2,1,1,1,0,0,0


Let's see BoW reward a repeated word. Here `the` appears twice, so its count is `2`.

In [8]:
demo = ["the cat sat on the mat"]
demo_counts = bow_vec.transform(demo)   # reuse the vocabulary learned above
present = {w: int(c) for w, c in zip(bow_vec.get_feature_names_out(), demo_counts.toarray()[0]) if c > 0}
print("Sentence:", demo[0])
print("Counts (words known to the vocab):", present)
print("\nNote: 'on' and 'mat' were not in our tiny corpus, so they are ignored (OOV).")

Sentence: the cat sat on the mat
Counts (words known to the vocab): {'cat': 1, 'sat': 1, 'the': 2}

Note: 'on' and 'mat' were not in our tiny corpus, so they are ignored (OOV).


**The catch:** common words rack up the biggest counts but carry the least information. In real text, `the`, `is`, `a` would dominate every document. We want to *down-weight* words that appear everywhere — that's TF-IDF.

---
## 4. TF-IDF — weight by how informative a word is

**TF-IDF = Term Frequency × Inverse Document Frequency.**

- **TF** — how often a word appears *in this document* (frequent here ⇒ relevant here).
- **IDF** — how *rare* the word is *across all documents* (in every doc ⇒ near-zero weight).

A word scores high only when it is frequent **here** *and* rare **elsewhere**.

In [9]:
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)
print("TF-IDF weights:")
df_tfidf

TF-IDF weights:


,and,cat,dog,ran,sat,the
doc0,0.000,0.518,0.000,0.000,0.681,0.518
doc1,0.000,0.000,0.518,0.681,0.000,0.518
doc2,0.681,0.518,0.518,0.000,0.000,0.000


Look at the IDF values scikit-learn computed. A word in **every** document gets the **smallest** IDF, so it is pushed toward zero.

In [10]:
idf = pd.Series(tfidf_vec.idf_, index=tfidf_vec.get_feature_names_out(), name="idf")
idf = idf.sort_values()
print("IDF per word (lower = more common = less informative):")
print(idf)

print("\n'the' appears in 2 of 3 docs -> low IDF -> down-weighted.")
print("'and', 'ran', 'sat' appear in just 1 doc -> high IDF -> emphasized.")

IDF per word (lower = more common = less informative):
cat    1.287682
dog    1.287682
the    1.287682
and    1.693147
ran    1.693147
sat    1.693147
Name: idf, dtype: float64

'the' appears in 2 of 3 docs -> low IDF -> down-weighted.
'and', 'ran', 'sat' appear in just 1 doc -> high IDF -> emphasized.


### BoW vs TF-IDF, side by side

Same corpus, same columns — only the numbers differ. Watch how the common word `the` shrinks relative to distinctive words.

In [11]:
compare = pd.concat(
    {"Bag of Words": df_bow, "TF-IDF": df_tfidf},
    axis=1,
)
print("Counts vs weighted scores:")
compare

Counts vs weighted scores:


Bag of Words                     TF-IDF                              \
              and cat dog ran sat the    and    cat    dog    ran    sat   
doc0            0   1   0   0   1   1  0.000  0.518  0.000  0.000  0.681   
doc1            0   0   1   1   0   1  0.000  0.000  0.518  0.681  0.000   
doc2            1   1   1   0   0   0  0.681  0.518  0.518  0.000  0.000   

             
        the  
doc0  0.518  
doc1  0.518  
doc2  0.000

---
## 5. Why it matters: a quick application

A common use of TF-IDF is **document similarity** (the backbone of search). We turn documents into TF-IDF vectors and measure the angle between them with **cosine similarity** — `1.0` means identical direction, `0.0` means nothing in common.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "the cat sat on the mat",
    "a cat was sitting on a mat",   # same topic, different words
    "the stock market crashed today",  # unrelated topic
]

vec = TfidfVectorizer(stop_words="english")   # drop common words like 'the','a','on'
X = vec.fit_transform(docs)

sim = cosine_similarity(X)
sim_df = pd.DataFrame(sim.round(2),
                      index=[f"doc{i}" for i in range(len(docs))],
                      columns=[f"doc{i}" for i in range(len(docs))])
print("Cosine similarity between documents:")
print(sim_df)

print("\ndoc0 vs doc1 (both about a cat on a mat):", round(sim[0,1], 2))
print("doc0 vs doc2 (cat vs stock market):       ", round(sim[0,2], 2))

Cosine similarity between documents:
      doc0  doc1  doc2
doc0  1.00  0.54   0.0
doc1  0.54  1.00   0.0
doc2  0.00  0.00   1.0

doc0 vs doc1 (both about a cat on a mat): 0.54
doc0 vs doc2 (cat vs stock market):        0.0


The two cat sentences come out **more similar** to each other than either is to the stock-market sentence — even though they share almost no identical words beyond `cat` and `mat`. That is TF-IDF doing useful work.

---
## 6. The shared blind spot

All three methods treat words as **independent symbols**. They never learn that words can be *related*, and they ignore word **order**.

In [13]:
pair = ["dog bites man", "man bites dog"]
v = CountVectorizer().fit(pair)
m = v.transform(pair).toarray()

print("Vocabulary:", list(v.get_feature_names_out()))
print("Vector for 'dog bites man':", m[0])
print("Vector for 'man bites dog':", m[1])
print("\nIdentical vectors -> opposite meaning. Word order is completely lost.")

Vocabulary: ['bites', 'dog', 'man']
Vector for 'dog bites man': [1 1 1]
Vector for 'man bites dog': [1 1 1]

Identical vectors -> opposite meaning. Word order is completely lost.


`king` and `queen` would be as unrelated as `king` and `banana`, and the two sentences above are indistinguishable. The fix is the next topic: **word embeddings** — dense vectors learned from data where similar words sit close together.

---
## 7. Your turn

Try these to cement the ideas:

1. Add a new document to `corpus` (e.g. `"the cat and the dog"`) and re-run the BoW and TF-IDF cells. How does the IDF of `the` change?
2. Set `TfidfVectorizer(ngram_range=(1, 2))` to include **bigrams**. Does `"dog bites man"` now differ from `"man bites dog"`? Why?
3. Replace the corpus with a few real sentences of your own and find the two most similar documents using the cosine-similarity code in Section 5.

In [14]:
# Scratch space for your experiments
# my_corpus = ["...", "...", "..."]
# v = TfidfVectorizer()
# X = v.fit_transform(my_corpus)
# pd.DataFrame(X.toarray().round(3), columns=v.get_feature_names_out())

---
### Recap

| Method | What the numbers are | Common words | Best for |
|---|---|---|---|
| **One-Hot** | 0 / 1 per word | treated equally | tiny vocab, labels |
| **Bag of Words** | word counts | over-weighted | quick baselines |
| **TF-IDF** | weighted scores | down-weighted | search, classification |

None capture **order** or **meaning** — that is the door into word embeddings, which we open next session.